### FEATURE SELECTION

Pipeline de 5 etapas para reduzir as ~400 features originais:

1. **Qualidade** — variância quase zero e cardinalidade problemática
2. **Evidência Univariada** — IV, KS (numéricas) | Qui² + V de Cramér (categóricas)
3. **Redundância** — correlação entre numéricas; pares redundantes → manter a de maior evidência|

In [0]:
dados = spark.sql("select * from fraud_detection_dev.silver.fraud_data_clean")

In [0]:
from pyspark.sql import functions as F

q60, q80 = dados.approxQuantile('TransactionDT', [0.60, 0.80], 0.01)

train_df = dados.filter(F.col('TransactionDT') <= q60)
test_df = dados.filter((F.col('TransactionDT') > q60) & (F.col('TransactionDT') <= q80))
val_df = dados.filter(F.col('TransactionDT') > q80)

dia = lambda dt: dt // 86400

for nome, d in [('Treino', train_df), ('Teste', test_df), ('Validacao', val_df)]:
    dt_min, dt_max = d.select(F.min('TransactionDT'), F.max('TransactionDT')).first()
    print(f"{nome:10} {d.count():>7} | dias {dia(dt_min)}-{dia(dt_max)} ({dia(dt_max) - dia(dt_min)} dias)")

In [0]:
# Converter para pandas e identificar colunas
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance

df = train_df.toPandas()

# Definir a coluna target
TARGET = 'target'

# Identificar colunas numéricas e categóricas (excluindo TransactionID e target)
all_cols = [c for c in df.columns if c not in ['TransactionID', TARGET]]
numeric_cols = [c for c in all_cols if df[c].dtype in ['int64', 'float64']]
categorical_cols = [c for c in all_cols if df[c].dtype == 'object']

print(f"Total de features: {len(all_cols)} | Numéricas: {len(numeric_cols)} | Categóricas: {len(categorical_cols)}")

#### Etapa 1 - Qualidade

Remoção de features com problemas estruturais:
- **Variância quase zero**: baixíssima variabilidade (numéricas com variância < 0.01)
- **Cardinalidade problemática**: constante (1 valor único) ou ID-like (razão único/não-nulo > 95%)
- **Quase constante**: valor mais frequente aparece em > 95% das linhas não-nulas

In [0]:
# Etapa 1: remover features com problemas estruturais
removed_quality = set()

for col in numeric_cols + categorical_cols:
    n_unique = df[col].nunique(dropna=True)
    n_non_null = df[col].notna().sum()
    
    # Constante (1 valor único)
    if n_unique == 1:
        removed_quality.add(col)
        continue
    
    # ID-like (alta cardinalidade)
    if n_unique / max(n_non_null, 1) > 0.95:
        removed_quality.add(col)
        continue
    
    # Valor mais frequente aparece em > 95% das linhas
    if n_non_null > 0:
        top_freq_pct = df[col].value_counts().iloc[0] / n_non_null
        if top_freq_pct > 0.95:
            removed_quality.add(col)
            continue
    
    # Variância quase zero (apenas numéricas)
    if col in numeric_cols:
        variance = df[col].var()
        if pd.notna(variance) and variance < 0.01:
            removed_quality.add(col)

features_q = [c for c in numeric_cols + categorical_cols if c not in removed_quality]
numeric_q = [c for c in features_q if c in numeric_cols]
categorical_q = [c for c in features_q if c in categorical_cols]

# Salvar lista de features removidas para consulta
removed_quality_list = sorted(list(removed_quality))

print(f"{len(removed_quality)} features removidas, {len(features_q)} restantes")

In [0]:
display(dados.select(removed_quality_list).summary())

#### Etapa 2 — Evidência Univariada

Medidas de poder preditivo individual de cada feature:
- **Numéricas**: Information Value (IV) e estatística de Kolmogorov-Smirnov (KS)
- **Categóricas**: Qui² + V de Cramér (associação com o target)

Score unificado para ranking: KS (numéricas) e V de Cramér (categóricas), ambos em escala 0–1.

In [0]:
# Etapa 2: medir poder preditivo de cada feature

def compute_ks(series, target):
    """Estatística de Kolmogorov-Smirnov"""
    data = pd.DataFrame({'val': series, 'target': target}).dropna()
    eventos = data[data['target'] == 1]['val']
    nao_eventos = data[data['target'] == 0]['val']
    if len(eventos) == 0 or len(nao_eventos) == 0:
        return 0.0
    return stats.ks_2samp(eventos, nao_eventos).statistic

def compute_cramers_v(series, target):
    """V de Cramér para associação categórica"""
    data = pd.DataFrame({'val': series, 'target': target}).dropna()
    if len(data) < 2 or data['val'].nunique() < 2:
        return 0.0
    
    contingencia = pd.crosstab(data['val'], data['target'])
    if contingencia.shape[0] < 2 or contingencia.shape[1] < 2:
        return 0.0
    
    chi2 = stats.chi2_contingency(contingencia)[0]
    n = contingencia.values.sum()
    min_dim = min(contingencia.shape) - 1
    return np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0.0

# Calcular evidência para cada feature
evidence_scores = {}

for col in numeric_q:
    evidence_scores[col] = compute_ks(df[col], df[TARGET])

for col in categorical_q:
    evidence_scores[col] = compute_cramers_v(df[col], df[TARGET])

evidence_df = pd.DataFrame([
    {'feature': k, 'evidence_score': v} 
    for k, v in evidence_scores.items()
]).sort_values('evidence_score', ascending=False).reset_index(drop=True)

print(f"Etapa 2 concluída: evidência univariada calculada para {len(evidence_df)} features")

In [0]:
display(evidence_df)

#### Etapa 3 — Redundância

Identificação e remoção de features altamente correlacionadas entre si:
- **Numéricas**: correlação de Pearson (|r| > 0.85)
- **Categóricas**: V de Cramér entre pares (> 0.85)
- Entre pares redundantes, **remover a de menor evidência univariada** (Etapa 2)

In [0]:
# Etapa 3: remover features redundantes
CORR_THRESHOLD = 0.85

ev_score = dict(zip(evidence_df['feature'], evidence_df['evidence_score']))
removed_redundancy = set()

# Remover correlações altas entre numéricas
if len(numeric_q) > 1:
    corr_matrix = df[numeric_q].corr().abs()
    
    for i in range(len(numeric_q)):
        for j in range(i + 1, len(numeric_q)):
            if corr_matrix.iloc[i, j] > CORR_THRESHOLD:
                col_i, col_j = numeric_q[i], numeric_q[j]
                # Manter a de maior evidência
                weaker = col_i if ev_score.get(col_i, 0) < ev_score.get(col_j, 0) else col_j
                removed_redundancy.add(weaker)

# Remover associações altas entre categóricas (V de Cramér)
for i in range(len(categorical_q)):
    for j in range(i + 1, len(categorical_q)):
        col_i, col_j = categorical_q[i], categorical_q[j]
        
        data = df[[col_i, col_j]].dropna()
        if len(data) < 2:
            continue
        
        contingencia = pd.crosstab(data[col_i], data[col_j])
        if contingencia.shape[0] < 2 or contingencia.shape[1] < 2:
            continue
        
        chi2 = stats.chi2_contingency(contingencia)[0]
        n = contingencia.values.sum()
        min_dim = min(contingencia.shape) - 1
        v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        
        if v > CORR_THRESHOLD:
            weaker = col_i if ev_score.get(col_i, 0) < ev_score.get(col_j, 0) else col_j
            removed_redundancy.add(weaker)

features_r = [c for c in features_q if c not in removed_redundancy]
numeric_r = [c for c in features_r if c in numeric_q]
categorical_r = [c for c in features_r if c in categorical_q]

# Salvar lista de features removidas para consulta
removed_redundancy_list = sorted(list(removed_redundancy))

print(f"Etapa 3 concluída: {len(removed_redundancy)} features redundantes removidas, {len(features_r)} restantes")

In [0]:
display(dados.select(removed_redundancy_list))

A variável **DeviceType** foi removida durante o processo. Entretanto, por ter a informaçao de origem da transaçao no sentido de mobile ou desktop, iremos colocar ela de volta no conjunto de dados e além disso, construir uma variável binária que nos diga se um determinado valor dessa variável está nulo ou nao. Pode ser que, por essa variável estar nula, deve ter algum padrao com a fraude.

In [0]:
removed_redundancy.discard('DeviceType')

features_r = [c for c in features_q if c not in removed_redundancy]
numeric_r = [c for c in features_r if c in numeric_q]
categorical_r = [c for c in features_r if c in categorical_q]
removed_redundancy_list = sorted(list(removed_redundancy))

%md
### Features Selecionadas

As features finais estão prontas para **Feature Engineering** e **Modelagem Definitiva**.

In [0]:
features_remove = list(removed_redundancy_list) + list(removed_quality_list)
df = df.drop(columns=features_remove, errors='ignore')

In [0]:
FREE_DOMAINS = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'live.com', 'icloud.com', 'aol.com', 'msn.com']

def apply_feature_engineering(df, agg_stats=None):
    df = df.copy()
    
    df['DeviceType_isnull'] = df['DeviceType'].isna().astype(int)
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_round'] = (df['TransactionAmt'] % 1 == 0).astype(int)
    df['transaction_hour'] = ((df['TransactionDT'] // 3600) % 24).astype(int)
    df['transaction_day'] = ((df['TransactionDT'] // 86400) % 7).astype(int)
    df['is_morning'] = ((df['transaction_hour'] >= 6) & (df['transaction_hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['transaction_hour'] >= 12) & (df['transaction_hour'] < 18)).astype(int)
    df['is_night'] = ((df['transaction_hour'] >= 18) | (df['transaction_hour'] < 6)).astype(int)
    df['is_weekend'] = (df['transaction_day'] >= 5).astype(int)
    df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
    df['P_email_free'] = df['P_emaildomain'].isin(FREE_DOMAINS).astype(int)
    df['R_email_free'] = df['R_emaildomain'].isin(FREE_DOMAINS).astype(int)
    
    if agg_stats is None:
        agg_stats = {
            'card1_tx_count': df.groupby('card1')['TransactionAmt'].count(),
            'card1_amt_mean': df.groupby('card1')['TransactionAmt'].mean(),
            'card1_amt_std': df.groupby('card1')['TransactionAmt'].std(),
            'card1_typical_device': df.groupby('card1')['DeviceType'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
            'addr1_tx_count': df.groupby('addr1')['TransactionAmt'].count(),
            'addr1_amt_mean': df.groupby('addr1')['TransactionAmt'].mean(),
        }
    
    df['card1_tx_count'] = df['card1'].map(agg_stats['card1_tx_count'])
    df['card1_amt_mean'] = df['card1'].map(agg_stats['card1_amt_mean'])
    df['card1_amt_std'] = df['card1'].map(agg_stats['card1_amt_std'])
    df['TransactionAmt_dev_card'] = df['TransactionAmt'] - df['card1_amt_mean']
    df['TransactionAmt_zscore_card'] = df['TransactionAmt_dev_card'] / (df['card1_amt_std'] + 1e-6)
    typical_device = df['card1'].map(agg_stats['card1_typical_device'])
    df['device_mismatch'] = ((df['DeviceType'] != typical_device) & df['DeviceType'].notna() & typical_device.notna()).astype(int)
    df['addr1_tx_count'] = df['addr1'].map(agg_stats['addr1_tx_count'])
    df['addr1_amt_mean'] = df['addr1'].map(agg_stats['addr1_amt_mean'])
    
    return df, agg_stats

new_numeric = ['DeviceType_isnull', 'TransactionAmt_log', 'TransactionAmt_round',
              'transaction_hour', 'transaction_day', 'is_morning', 'is_afternoon', 'is_night', 'is_weekend',
              'card1_tx_count', 'card1_amt_mean', 'card1_amt_std',
              'TransactionAmt_dev_card', 'TransactionAmt_zscore_card', 'device_mismatch',
              'addr1_tx_count', 'addr1_amt_mean',
              'email_match', 'P_email_free', 'R_email_free']

for f in new_numeric:
    if f not in features_r:
        features_r.append(f)
    if f not in numeric_r:
        numeric_r.append(f)

In [0]:
features_drop = list(removed_redundancy_list) + list(removed_quality_list) + ['ingestion_timestamp']

train_pd = train_df.toPandas().drop(columns=features_drop, errors='ignore')
test_pd = test_df.toPandas().drop(columns=features_drop, errors='ignore')
val_pd = val_df.toPandas().drop(columns=features_drop, errors='ignore')

train_fe, agg_stats = apply_feature_engineering(train_pd)
test_fe, _ = apply_feature_engineering(test_pd, agg_stats)
val_fe, _ = apply_feature_engineering(val_pd, agg_stats)

train_s = spark.createDataFrame(train_fe)
test_s = spark.createDataFrame(test_fe)
val_s = spark.createDataFrame(val_fe)

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid='keep') for c in categorical_r]

assembler = VectorAssembler(
    inputCols=numeric_r + [f"{c}_idx" for c in categorical_r],
    outputCol='features',
    handleInvalid='keep'
)

pipeline = Pipeline(stages=indexers + [assembler])
pipeline_model = pipeline.fit(train_s)

train_final = pipeline_model.transform(train_s).select('TransactionID', 'target', 'features')
test_final = pipeline_model.transform(test_s).select('TransactionID', 'target', 'features')
val_final = pipeline_model.transform(val_s).select('TransactionID', 'target', 'features')

print(f"Train: {train_final.count()} | Test: {test_final.count()} | Val: {val_final.count()}")

In [0]:
for nome, df in [('train', train_final), ('test', test_final), ('val', val_final)]:
    table = f'fraud_detection_dev.silver.fraud_{nome}_features'
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table)
    print(f'{table}: {df.count()} registros')